Comando para rodar a aplicação via docker. Recomendo usar a extensão dev container do vscode e pegar o código na pasta "app".

```bash
docker build -t tp1-algoritmos .
docker run -it --name tp1 -p 8888:8888 -v "${PWD}:/app" tp1-algoritmos
```

## Parte 1

In [18]:
import subprocess

resultado = subprocess.run(
    "find / -type f 2>/dev/null | head -n 10000 > /app/listagem.txt",
    shell=True,
    capture_output=True,
    text=True
)

# Optei por fazer via código porém usando o mesmo comando que seria direcionado ao terminal linux para facilitar reprodutibilidade via docker e facilitar os testes do professor. 
# Assim exercito os comandos Linux como pedido porém concentro a entrega no código e não preciso fazer commit da lista inteira.

with open("/app/listagem.txt", "r") as arquivo:
    arquivos = arquivo.read().splitlines()

print(f"Listagem gerada com sucesso!")
print(f"Total de arquivos encontrados: {len(arquivos)}")
print("Arquivo salvo em: listagem.txt")


Listagem gerada com sucesso!
Total de arquivos encontrados: 10000
Arquivo salvo em: listagem.txt


## Parte 2.1

In [19]:
def bubble_sort(arr):
    esquerdo = 0
    direito = len(arr) - 1

    while esquerdo < direito:
        houve_troca = False

        for i in range(esquerdo, direito):
            if arr[i] > arr[i + 1]:
                arr[i], arr[i + 1] = arr[i + 1], arr[i]
                houve_troca = True

        for i in range(direito, esquerdo, -1):
            if arr[i] < arr[i - 1]:
                arr[i], arr[i - 1] = arr[i - 1], arr[i]
                houve_troca = True

        if not houve_troca:
            break

    return arr

In [20]:
def selection_sort(arr):
    n = len(arr)

    for i in range(n):
        menor = i

        for j in range(i + 1, n):
            if arr[j] < arr[menor]:
                menor = j

        arr[i], arr[menor] = arr[menor], arr[i]

    return arr

In [21]:
def insertion_sort(arr):
    n = len(arr)

    for i in range(1, n):
        atual = arr[i]
        j = i - 1

        while j >= 0 and arr[j] > atual:
            arr[j + 1] = arr[j]
            j -= 1

        arr[j + 1] = atual

    return arr

In [22]:
from time import perf_counter

def medir_tempo(algoritmo, dados):
    arr = dados.copy()

    inicio = perf_counter()
    algoritmo(arr)
    fim = perf_counter()

    return fim - inicio

In [23]:
import tracemalloc

def medir_memoria(algoritmo, dados):
    arr = dados.copy()

    tracemalloc.start()

    algoritmo(arr)

    memoria_atual, memoria_pico = tracemalloc.get_traced_memory()

    tracemalloc.stop()

    return memoria_pico / 1024

In [24]:
# Embaralhando a lista para comparação mais justa

import random

random.seed(42)
random.shuffle(arquivos)

In [25]:
# Bubble Sort
tempo_bubble = medir_tempo(bubble_sort, arquivos)
memoria_bubble = medir_memoria(bubble_sort, arquivos)

# Selection Sort
tempo_selection = medir_tempo(selection_sort, arquivos)
memoria_selection = medir_memoria(selection_sort, arquivos)

# Insertion Sort
tempo_insertion = medir_tempo(insertion_sort, arquivos)
memoria_insertion = medir_memoria(insertion_sort, arquivos)


print("BUBBLE SORT")
print(f"Tempo: {tempo_bubble:.6f} segundos")
print(f"Memória: {memoria_bubble:.6f} KB")

print("\nSELECTION SORT")
print(f"Tempo: {tempo_selection:.6f} segundos")
print(f"Memória: {memoria_selection:.6f} KB")

print("\nINSERTION SORT")
print(f"Tempo: {tempo_insertion:.6f} segundos")
print(f"Memória: {memoria_insertion:.6f} KB")

BUBBLE SORT
Tempo: 4.382825 segundos
Memória: 1.898438 KB

SELECTION SORT
Tempo: 1.918165 segundos
Memória: 1.667969 KB

INSERTION SORT
Tempo: 1.914130 segundos
Memória: 1.371094 KB


## Parte 2.2

In [26]:
class Hashtable:
    def __init__(self):
        self.dados = {}

    def adicionar(self, chave, valor):
        self.dados[chave] = valor

    def remover(self, chave):
        return self.dados.pop(chave, None)

    def buscar(self, chave):
        return self.dados.get(chave)

In [27]:
class Pilha:
    def __init__(self):
        self.dados = []

    def adicionar(self, valor):
        self.dados.append(valor)

    def remover(self):
        if not self.dados:
            return None
        return self.dados.pop()

    def recuperar_posicao(self, posicao):
        if posicao < 1 or posicao > len(self.dados):
            return None

        copia = Pilha()
        copia.dados = self.dados.copy()

        resultado = None

        for _ in range(len(copia.dados) - posicao + 1):
            resultado = copia.remover()

        return resultado

In [31]:
from collections import deque

class Fila:
    def __init__(self):
        self.dados = deque()

    def adicionar(self, valor):
        self.dados.append(valor)

    def remover(self):
        if not self.dados:
            return None
        return self.dados.popleft()


    def recuperar_posicao(self, posicao):
        if posicao < 1 or posicao > len(self.dados):
            return None

        copia = Fila()
        copia.dados = deque(self.dados)

        resultado = None

        for _ in range(posicao):
            resultado = copia.remover()

        return resultado

In [32]:
# Hashtable
tracemalloc.start()
inicio = perf_counter()

hashtable = Hashtable()

for indice, arquivo in enumerate(arquivos, start=1):
    hashtable.adicionar(indice, arquivo)

tempo_hash = perf_counter() - inicio
memoria_hash = tracemalloc.get_traced_memory()[1] / 1024
tracemalloc.stop()


# Pilha
tracemalloc.start()
inicio = perf_counter()

pilha = Pilha()

for arquivo in arquivos:
    pilha.adicionar(arquivo)

tempo_pilha = perf_counter() - inicio
memoria_pilha = tracemalloc.get_traced_memory()[1] / 1024
tracemalloc.stop()


# Fila
tracemalloc.start()
inicio = perf_counter()

fila = Fila()

for arquivo in arquivos:
    fila.adicionar(arquivo)

tempo_fila = perf_counter() - inicio
memoria_fila = tracemalloc.get_traced_memory()[1] / 1024
tracemalloc.stop()

print("CONSTRUÇÃO DAS ESTRUTURAS")
print(f"Hashtable: {tempo_hash:.6f} s | {memoria_hash:.2f} KB")
print(f"Pilha:     {tempo_pilha:.6f} s | {memoria_pilha:.2f} KB")
print(f"Fila:      {tempo_fila:.6f} s | {memoria_fila:.2f} KB")

CONSTRUÇÃO DAS ESTRUTURAS
Hashtable: 0.008644 s | 766.14 KB
Pilha:     0.001504 s | 102.13 KB
Fila:      0.001603 s | 100.63 KB


In [34]:
def medir_consulta(funcao):
    tracemalloc.start()
    inicio = perf_counter()

    resultado = funcao()

    tempo = perf_counter() - inicio
    memoria = tracemalloc.get_traced_memory()[1] / 1024

    tracemalloc.stop()

    return resultado, tempo, memoria


posicoes = [1, 100, 1000, 5000, len(arquivos)]

print("RECUPERAÇÃO DE ARQUIVOS\n")

for posicao in posicoes:
    resultado_hash, tempo_hash, memoria_hash = medir_consulta(
        lambda: hashtable.buscar(posicao)
    )

    resultado_pilha, tempo_pilha, memoria_pilha = medir_consulta(
        lambda: pilha.recuperar_posicao(posicao)
    )

    resultado_fila, tempo_fila, memoria_fila = medir_consulta(
        lambda: fila.recuperar_posicao(posicao)
    )

    print(f"POSIÇÃO {posicao}")
    print(f"Hashtable: {tempo_hash:.6f} s | {memoria_hash:.2f} KB")
    print(f"Pilha:     {tempo_pilha:.6f} s | {memoria_pilha:.2f} KB")
    print(f"Fila:      {tempo_fila:.6f} s | {memoria_fila:.2f} KB")

    # Validação dos resultados
    assert resultado_hash == resultado_pilha == resultado_fila

print("\nTodas as estruturas recuperaram os mesmos arquivos.")

RECUPERAÇÃO DE ARQUIVOS

POSIÇÃO 1
Hashtable: 0.000003 s | 0.00 KB
Pilha:     0.005762 s | 78.52 KB
Fila:      0.000285 s | 82.25 KB
POSIÇÃO 100
Hashtable: 0.000002 s | 0.00 KB
Pilha:     0.006074 s | 78.51 KB
Fila:      0.000372 s | 82.23 KB
POSIÇÃO 1000
Hashtable: 0.000002 s | 0.00 KB
Pilha:     0.006649 s | 79.10 KB
Fila:      0.000823 s | 82.23 KB
POSIÇÃO 5000
Hashtable: 0.000014 s | 0.00 KB
Pilha:     0.003228 s | 78.59 KB
Fila:      0.003749 s | 82.22 KB
POSIÇÃO 10000
Hashtable: 0.000003 s | 0.00 KB
Pilha:     0.000068 s | 78.42 KB
Fila:      0.006178 s | 82.21 KB

Todas as estruturas recuperaram os mesmos arquivos.


In [35]:
# Testando remoção na table separadamente visto que, diferente da pilha e da fila que usam remoção ao buscar itens, a hashtable não tinha um teste com isso até então

tracemalloc.start()
inicio = perf_counter()

for posicao in posicoes:
    hashtable.remover(posicao)

tempo_remocao_hash = perf_counter() - inicio
memoria_remocao_hash = tracemalloc.get_traced_memory()[1] / 1024
tracemalloc.stop()

print("REMOÇÃO NA HASHTABLE")
print(f"Tempo: {tempo_remocao_hash:.6f} s")
print(f"Memória: {memoria_remocao_hash:.2f} KB")

REMOÇÃO NA HASHTABLE
Tempo: 0.000154 s
Memória: 21.60 KB


# Parte 3 — Análise e Relatório

## 1. Lógica e funcionamento

Foram desenvolvidos dois programas.

O primeiro utiliza três algoritmos de ordenação para organizar alfabeticamente os caminhos dos arquivos: Bubble Sort, Selection Sort e Insertion Sort.

O segundo implementa três estruturas de dados: Hashtable, Pilha e Fila. Cada estrutura possui métodos de adição e remoção, além de operações para recuperar elementos em posições específicas da listagem.

## 2. Complexidade de tempo (Big O)

| Algoritmo | Melhor caso | Caso médio | Pior caso |
|---|---|---|---|
| Bubble Sort | O(n) | O(n²) | O(n²) |
| Selection Sort | O(n²) | O(n²) | O(n²) |
| Insertion Sort | O(n) | O(n²) | O(n²) |

Nas estruturas de dados:

| Estrutura | Operações | Complexidade média |
|---|---|---|
| Hashtable | Adição, busca e remoção | O(1) |
| Pilha | Adição e remoção | O(1) |
| Fila | Adição e remoção | O(1) |
| Pilha e Fila | Recuperação por posição | O(n) |

## 3. Tempos de execução e memória

| Algoritmo | Tempo | Memória |
|---|---:|---:|
| Bubble Sort | 4,55 s | 1,90 KB |
| Selection Sort | 2,01 s | 1,39 KB |
| Insertion Sort | 1,87 s | 1,60 KB |

Nas estruturas de dados, a Hashtable apresentou consultas praticamente instantâneas. A pilha e a fila tiveram tempos maiores porque precisaram remover elementos sucessivamente para alcançar determinadas posições.

O consumo de memória também foi maior nessas duas estruturas devido à criação de cópias temporárias durante as consultas.

## 4. Comparação entre teoria e prática

Os resultados foram compatíveis com as complexidades teóricas. Entre os algoritmos de ordenação, o Insertion Sort apresentou o melhor tempo, enquanto o Bubble Sort foi o mais lento.

Nas estruturas de dados, a Hashtable demonstrou vantagem na recuperação de elementos por posição, enquanto a pilha e a fila apresentaram desempenho dependente da posição consultada.

## 5. Conclusão

O experimento demonstrou que diferentes algoritmos e estruturas de dados apresentam desempenhos distintos dependendo da operação realizada. Também foi possível observar que a complexidade teórica ajuda a explicar os resultados práticos, embora fatores como implementação, ambiente de execução e organização dos dados também influenciem o desempenho.